In [1]:
from pathlib import Path
import os, sys, io, json, zipfile, shutil, subprocess, random, math
from datetime import datetime

# =========================================================
# INSTALLS
# =========================================================
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "numpy", "pandas", "pillow", "matplotlib", "tqdm"],
    check=True
)

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

# =========================================================
# SETTINGS
# =========================================================
MODE = "train"                       # "train" or "status"
TRAIN_REGRESSOR_IF_MISSING = True
FORCE_RETRAIN_REGRESSOR = False

# Regressor settings
LM_REG_EPOCHS = 3
LM_REG_BATCH_SIZE = 64
LM_REG_LR = 1e-3
LM_REG_WEIGHT_DECAY = 1e-4
LM_REG_NUM_WORKERS = 2
LM_VAL_FRAC = 0.1
LM_RANDOM_SEED = 42

# Landmark regularization settings used inside GAN training
LM_LAMBDA = 0.25
LM_Z_MARGIN = 2.5

# Modified GAN training: keep same safe settings as baseline, but short run first
MOD_KIMG = 5
MOD_TICK = 1
MOD_SNAP = 1
MOD_METRICS = "none"
MOD_SEED = 42

CFG = "stylegan3-r"
GAMMA = 1.0
CBASE = 16384
GLR = 0.0025
DLR = 0.0025
AUG = "noaug"
MIRROR = 0
BATCH = 2
BATCH_GPU = 1
WORKERS = 1
MBSTD_GROUP = 1

# =========================================================
# PATHS
# =========================================================
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

dataset_zip = ROOT / "data" / "stylegan3_zip" / "celeba_256x256.zip"
stats_npz = ROOT / "data" / "manifests" / "real_landmark_stats.npz"
stats_csv = ROOT / "data" / "manifests" / "real_landmark_features.csv"

stylegan3_repo = ROOT / "third_party" / "stylegan3"
train_py = stylegan3_repo / "train.py"
loss_py = stylegan3_repo / "training" / "loss.py"
loss_backup = stylegan3_repo / "training" / "loss_original_backup.py"

baseline_root = ROOT / "checkpoints" / "baseline_stylegan3r"
modified_root = ROOT / "checkpoints" / "landmark_reg"
modified_root.mkdir(parents=True, exist_ok=True)

models_dir = ROOT / "checkpoints" / "aux_models"
models_dir.mkdir(parents=True, exist_ok=True)
reg_ckpt = models_dir / "landmark_feature_regressor.pt"

logs_dir = ROOT / "logs" / "landmark_reg"
logs_dir.mkdir(parents=True, exist_ok=True)

cache_dir = ROOT / ".cache"
(cache_dir / "torch_extensions").mkdir(parents=True, exist_ok=True)
(cache_dir / "dnnlib").mkdir(parents=True, exist_ok=True)

# CUDA/cache env
os.environ["TORCH_EXTENSIONS_DIR"] = str(cache_dir / "torch_extensions")
os.environ["DNNLIB_CACHE_DIR"] = str(cache_dir / "dnnlib")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:64"
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"

# These will be read by the patched loss.py
os.environ["LM_REG_ENABLE"] = "1"
os.environ["LM_REG_CKPT"] = str(reg_ckpt)
os.environ["LM_REG_LAMBDA"] = str(LM_LAMBDA)
os.environ["LM_REG_Z_MARGIN"] = str(LM_Z_MARGIN)

# =========================================================
# PRECHECKS
# =========================================================
assert dataset_zip.exists(), f"Missing dataset ZIP: {dataset_zip}"
assert stats_npz.exists(), f"Missing stats NPZ from Notebook 06: {stats_npz}"
assert stats_csv.exists(), f"Missing feature CSV from Notebook 06: {stats_csv}"
assert stylegan3_repo.exists(), f"Missing StyleGAN3 repo: {stylegan3_repo}"
assert train_py.exists(), f"Missing train.py: {train_py}"
assert loss_py.exists(), f"Missing training/loss.py: {loss_py}"

# =========================================================
# HELPERS
# =========================================================
def latest_run_dir(root: Path):
    runs = [p for p in root.iterdir() if p.is_dir()] if root.exists() else []
    runs = sorted(runs, key=lambda p: p.stat().st_mtime, reverse=True)
    return runs[0] if runs else None

def latest_snapshot(run_dir: Path):
    if run_dir is None or not run_dir.exists():
        return None
    snaps = sorted(run_dir.glob("network-snapshot-*.pkl"))
    return snaps[-1] if snaps else None

def latest_fakes(run_dir: Path):
    if run_dir is None or not run_dir.exists():
        return None
    imgs = sorted(run_dir.glob("fakes*.png"))
    return imgs[-1] if imgs else None

def tail_jsonl(path: Path, n=5):
    if not path.exists():
        return []
    lines = path.read_text(encoding="utf-8", errors="ignore").strip().splitlines()
    out = []
    for line in lines[-n:]:
        try:
            out.append(json.loads(line))
        except Exception:
            out.append({"raw": line})
    return out

def run_cmd(cmd, cwd=None, capture=False, env=None):
    if capture:
        return subprocess.check_output(cmd, cwd=cwd, text=True, env=env)
    print("\n[RUN]", " ".join(str(x) for x in cmd))
    subprocess.run(cmd, cwd=cwd, check=True, env=env)

def stream_process(cmd, cwd=None, logfile=None, env=None):
    print("\n[RUN]")
    print(" ".join(str(x) for x in cmd))
    with subprocess.Popen(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True,
        env=env,
    ) as proc:
        with open(logfile, "a", encoding="utf-8") if logfile else open(os.devnull, "w") as logf:
            for line in proc.stdout:
                print(line, end="")
                if logfile:
                    logf.write(line)
            proc.wait()
            if proc.returncode != 0:
                raise subprocess.CalledProcessError(proc.returncode, cmd)

# =========================================================
# LOAD REAL LANDMARK STATS
# =========================================================
npz = np.load(stats_npz, allow_pickle=True)
feature_names = [str(x) for x in npz["feature_names"].tolist()]
real_mean = np.asarray(npz["mean"], dtype=np.float32)
real_std = np.asarray(npz["std"], dtype=np.float32)

print("Feature names:", feature_names)
print("Stats file:", stats_npz)
print("CSV file:", stats_csv)

# =========================================================
# LANDMARK FEATURE REGRESSOR
# =========================================================
class LandmarkFeatureRegressor(nn.Module):
    def __init__(self, n_outputs: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128), nn.ReLU(inplace=True),
            nn.Linear(128, n_outputs),
        )

    def forward(self, x):
        x = self.net(x)
        x = self.head(x)
        return x

class ZipFeatureDataset(Dataset):
    def __init__(self, zip_path: Path, csv_path: Path):
        self.zip_path = zip_path
        self.df = pd.read_csv(csv_path)
        needed = ["image"] + feature_names
        for c in needed:
            assert c in self.df.columns, f"Missing column '{c}' in {csv_path}"
        self.image_names = self.df["image"].tolist()
        self.targets = self.df[feature_names].to_numpy(dtype=np.float32)

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        inner_name = self.image_names[idx]
        target = self.targets[idx]
        with zipfile.ZipFile(self.zip_path, "r") as zf:
            data = zf.read(inner_name)
        img = Image.open(io.BytesIO(data)).convert("RGB")
        arr = np.asarray(img, dtype=np.float32) / 255.0
        arr = np.transpose(arr, (2, 0, 1))
        x = torch.from_numpy(arr)
        y = torch.from_numpy(target)
        return x, y

def train_regressor():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Regressor training device:", device)

    ds = ZipFeatureDataset(dataset_zip, stats_csv)
    n_total = len(ds)
    n_val = max(1, int(n_total * LM_VAL_FRAC))
    n_train = n_total - n_val

    g = torch.Generator().manual_seed(LM_RANDOM_SEED)
    train_ds, val_ds = random_split(ds, [n_train, n_val], generator=g)

    train_loader = DataLoader(
        train_ds, batch_size=LM_REG_BATCH_SIZE, shuffle=True,
        num_workers=LM_REG_NUM_WORKERS, pin_memory=torch.cuda.is_available()
    )
    val_loader = DataLoader(
        val_ds, batch_size=LM_REG_BATCH_SIZE, shuffle=False,
        num_workers=LM_REG_NUM_WORKERS, pin_memory=torch.cuda.is_available()
    )

    model = LandmarkFeatureRegressor(n_outputs=len(feature_names)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LM_REG_LR, weight_decay=LM_REG_WEIGHT_DECAY)
    loss_fn = nn.SmoothL1Loss()

    best_val = float("inf")
    history = []

    mean_t = torch.tensor(real_mean, device=device)
    std_t = torch.tensor(real_std, device=device)

    for epoch in range(1, LM_REG_EPOCHS + 1):
        model.train()
        train_loss_sum = 0.0
        train_count = 0

        for x, y in tqdm(train_loader, desc=f"Regressor train epoch {epoch}/{LM_REG_EPOCHS}"):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            # normalize targets for stable regression
            y_norm = (y - mean_t) / (std_t + 1e-8)

            pred = model(x)
            loss = loss_fn(pred, y_norm)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

            train_loss_sum += float(loss.item()) * x.size(0)
            train_count += x.size(0)

        model.eval()
        val_loss_sum = 0.0
        val_count = 0
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                y_norm = (y - mean_t) / (std_t + 1e-8)
                pred = model(x)
                loss = loss_fn(pred, y_norm)
                val_loss_sum += float(loss.item()) * x.size(0)
                val_count += x.size(0)

        train_loss = train_loss_sum / max(train_count, 1)
        val_loss = val_loss_sum / max(val_count, 1)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
        print(f"Epoch {epoch}: train_loss={train_loss:.6f}  val_loss={val_loss:.6f}")

        if val_loss < best_val:
            best_val = val_loss
            ckpt = {
                "model_state_dict": model.state_dict(),
                "feature_names": feature_names,
                "mean": real_mean,
                "std": real_std,
                "history": history,
            }
            torch.save(ckpt, reg_ckpt)
            print("Saved best regressor to:", reg_ckpt)

    hist_path = logs_dir / "landmark_regressor_history.json"
    hist_path.write_text(json.dumps(history, indent=2), encoding="utf-8")
    print("Saved history:", hist_path)

if (not reg_ckpt.exists()) or FORCE_RETRAIN_REGRESSOR:
    if not TRAIN_REGRESSOR_IF_MISSING:
        raise RuntimeError(f"Missing regressor checkpoint: {reg_ckpt}")
    train_regressor()
else:
    print("Using existing regressor checkpoint:", reg_ckpt)

assert reg_ckpt.exists(), f"Regressor checkpoint not found: {reg_ckpt}"

# =========================================================
# PATCH STYLEGAN3 LOSS.PY
# =========================================================
if not loss_backup.exists():
    shutil.copy2(loss_py, loss_backup)
    print("Backed up original loss.py to:", loss_backup)
else:
    print("Backup already exists:", loss_backup)

patched_loss_code = r'''
# Copyright (c) 2021, NVIDIA CORPORATION & AFFILIATES. All rights reserved.
"""Loss functions with optional landmark regularization."""
import os
import numpy as np
import torch
import torch.nn as nn
from torch_utils import training_stats
from torch_utils.ops import conv2d_gradfix
from torch_utils.ops import upfirdn2d

#----------------------------------------------------------------------------

class Loss:
    def accumulate_gradients(self, phase, real_img, real_c, gen_z, gen_c, gain, cur_nimg):
        raise NotImplementedError()

#----------------------------------------------------------------------------

class LandmarkFeatureRegressor(nn.Module):
    def __init__(self, n_outputs):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128), nn.ReLU(inplace=True),
            nn.Linear(128, n_outputs),
        )

    def forward(self, x):
        x = self.net(x)
        x = self.head(x)
        return x

#----------------------------------------------------------------------------

class StyleGAN2Loss(Loss):
    def __init__(self, device, G, D, augment_pipe=None, r1_gamma=10, style_mixing_prob=0, pl_weight=0, pl_batch_shrink=2, pl_decay=0.01, pl_no_weight_grad=False, blur_init_sigma=0, blur_fade_kimg=0):
        super().__init__()
        self.device = device
        self.G = G
        self.D = D
        self.augment_pipe = augment_pipe
        self.r1_gamma = r1_gamma
        self.style_mixing_prob = style_mixing_prob
        self.pl_weight = pl_weight
        self.pl_batch_shrink = pl_batch_shrink
        self.pl_decay = pl_decay
        self.pl_no_weight_grad = pl_no_weight_grad
        self.pl_mean = torch.zeros([], device=device)
        self.blur_init_sigma = blur_init_sigma
        self.blur_fade_kimg = blur_fade_kimg

        # Optional landmark regularization
        self.lm_reg_enable = os.environ.get("LM_REG_ENABLE", "0") == "1"
        self.lm_reg_lambda = float(os.environ.get("LM_REG_LAMBDA", "0.0"))
        self.lm_reg_z_margin = float(os.environ.get("LM_REG_Z_MARGIN", "2.5"))
        self.lm_regressor = None
        self.lm_mean = None
        self.lm_std = None

        if self.lm_reg_enable:
            ckpt_path = os.environ.get("LM_REG_CKPT", "")
            if ckpt_path and os.path.isfile(ckpt_path):
                ckpt = torch.load(ckpt_path, map_location=device)
                n_outputs = len(ckpt["feature_names"])
                self.lm_regressor = LandmarkFeatureRegressor(n_outputs=n_outputs).to(device)
                self.lm_regressor.load_state_dict(ckpt["model_state_dict"], strict=True)
                self.lm_regressor.eval()
                for p in self.lm_regressor.parameters():
                    p.requires_grad = False
                self.lm_mean = torch.tensor(ckpt["mean"], device=device, dtype=torch.float32)
                self.lm_std = torch.tensor(ckpt["std"], device=device, dtype=torch.float32)
            else:
                self.lm_reg_enable = False

    def run_G(self, z, c, update_emas=False):
        ws = self.G.mapping(z, c, update_emas=update_emas)
        if self.style_mixing_prob > 0:
            with torch.autograd.profiler.record_function('style_mixing'):
                cutoff = torch.empty([], dtype=torch.int64, device=ws.device).random_(1, ws.shape[1])
                cutoff = torch.where(torch.rand([], device=ws.device) < self.style_mixing_prob, cutoff, torch.full_like(cutoff, ws.shape[1]))
                ws[:, cutoff:] = self.G.mapping(torch.randn_like(z), c, update_emas=False)[:, cutoff:]
        img = self.G.synthesis(ws, update_emas=update_emas)
        return img, ws

    def run_D(self, img, c, blur_sigma=0, update_emas=False):
        blur_size = np.floor(blur_sigma * 3)
        if blur_size > 0:
            with torch.autograd.profiler.record_function('blur'):
                f = torch.arange(-blur_size, blur_size + 1, device=img.device).div(blur_sigma).square().neg().exp2()
                img = upfirdn2d.filter2d(img, f / f.sum())
        if self.augment_pipe is not None:
            img = self.augment_pipe(img)
        logits = self.D(img, c, update_emas=update_emas)
        return logits

    def run_landmark_reg(self, img):
        if (not self.lm_reg_enable) or (self.lm_regressor is None):
            return torch.zeros([], device=img.device)

        # StyleGAN images are in [-1,1]; regressor was trained on [0,1]
        x = ((img + 1.0) * 0.5).clamp(0, 1)
        pred_z = self.lm_regressor(x)  # predicts normalized landmark features
        # Since regressor was trained on z-scored targets, plausible faces should stay within margin
        penalty = torch.relu(torch.abs(pred_z) - self.lm_reg_z_margin).square().mean(dim=1)
        training_stats.report('Loss/G/landmark_reg_raw', penalty)
        return penalty

    def accumulate_gradients(self, phase, real_img, real_c, gen_z, gen_c, gain, cur_nimg):
        assert phase in ['Gmain', 'Greg', 'Gboth', 'Dmain', 'Dreg', 'Dboth']
        if self.pl_weight == 0:
            phase = {'Greg': 'none', 'Gboth': 'Gmain'}.get(phase, phase)
        if self.r1_gamma == 0:
            phase = {'Dreg': 'none', 'Dboth': 'Dmain'}.get(phase, phase)
        blur_sigma = max(1 - cur_nimg / (self.blur_fade_kimg * 1e3), 0) * self.blur_init_sigma if self.blur_fade_kimg > 0 else 0

        # Gmain: maximize logits for generated images + optional landmark regularization
        if phase in ['Gmain', 'Gboth']:
            with torch.autograd.profiler.record_function('Gmain_forward'):
                gen_img, _gen_ws = self.run_G(gen_z, gen_c)
                gen_logits = self.run_D(gen_img, gen_c, blur_sigma=blur_sigma)
                training_stats.report('Loss/scores/fake', gen_logits)
                training_stats.report('Loss/signs/fake', gen_logits.sign())
                loss_Gadv = torch.nn.functional.softplus(-gen_logits)
                training_stats.report('Loss/G/adv', loss_Gadv)

                if self.lm_reg_enable and self.lm_reg_lambda > 0:
                    lm_penalty = self.run_landmark_reg(gen_img)
                    loss_Gmain = loss_Gadv + self.lm_reg_lambda * lm_penalty
                else:
                    lm_penalty = torch.zeros_like(loss_Gadv)
                    loss_Gmain = loss_Gadv

                training_stats.report('Loss/G/landmark_reg', lm_penalty)
                training_stats.report('Loss/G/loss', loss_Gmain)

            with torch.autograd.profiler.record_function('Gmain_backward'):
                loss_Gmain.mean().mul(gain).backward()

        # Gpl
        if phase in ['Greg', 'Gboth']:
            with torch.autograd.profiler.record_function('Gpl_forward'):
                batch_size = gen_z.shape[0] // self.pl_batch_shrink
                gen_img, gen_ws = self.run_G(gen_z[:batch_size], gen_c[:batch_size])
                pl_noise = torch.randn_like(gen_img) / np.sqrt(gen_img.shape[2] * gen_img.shape[3])
                with torch.autograd.profiler.record_function('pl_grads'), conv2d_gradfix.no_weight_gradients(self.pl_no_weight_grad):
                    pl_grads = torch.autograd.grad(outputs=[(gen_img * pl_noise).sum()], inputs=[gen_ws], create_graph=True, only_inputs=True)[0]
                pl_lengths = pl_grads.square().sum(2).mean(1).sqrt()
                pl_mean = self.pl_mean.lerp(pl_lengths.mean(), self.pl_decay)
                self.pl_mean.copy_(pl_mean.detach())
                pl_penalty = (pl_lengths - pl_mean).square()
                training_stats.report('Loss/pl_penalty', pl_penalty)
                loss_Gpl = pl_penalty * self.pl_weight
                training_stats.report('Loss/G/reg', loss_Gpl)
            with torch.autograd.profiler.record_function('Gpl_backward'):
                loss_Gpl.mean().mul(gain).backward()

        # Dmain on fakes
        loss_Dgen = 0
        if phase in ['Dmain', 'Dboth']:
            with torch.autograd.profiler.record_function('Dgen_forward'):
                gen_img, _gen_ws = self.run_G(gen_z, gen_c, update_emas=True)
                gen_logits = self.run_D(gen_img, gen_c, blur_sigma=blur_sigma, update_emas=True)
                training_stats.report('Loss/scores/fake', gen_logits)
                training_stats.report('Loss/signs/fake', gen_logits.sign())
                loss_Dgen = torch.nn.functional.softplus(gen_logits)
            with torch.autograd.profiler.record_function('Dgen_backward'):
                loss_Dgen.mean().mul(gain).backward()

        # Dmain / Dr1 on reals
        if phase in ['Dmain', 'Dreg', 'Dboth']:
            name = 'Dreal' if phase == 'Dmain' else 'Dr1' if phase == 'Dreg' else 'Dreal_Dr1'
            with torch.autograd.profiler.record_function(name + '_forward'):
                real_img_tmp = real_img.detach().requires_grad_(phase in ['Dreg', 'Dboth'])
                real_logits = self.run_D(real_img_tmp, real_c, blur_sigma=blur_sigma)
                training_stats.report('Loss/scores/real', real_logits)
                training_stats.report('Loss/signs/real', real_logits.sign())

                loss_Dreal = 0
                if phase in ['Dmain', 'Dboth']:
                    loss_Dreal = torch.nn.functional.softplus(-real_logits)
                    training_stats.report('Loss/D/loss', loss_Dgen + loss_Dreal)

                loss_Dr1 = 0
                if phase in ['Dreg', 'Dboth']:
                    with torch.autograd.profiler.record_function('r1_grads'), conv2d_gradfix.no_weight_gradients():
                        r1_grads = torch.autograd.grad(outputs=[real_logits.sum()], inputs=[real_img_tmp], create_graph=True, only_inputs=True)[0]
                    r1_penalty = r1_grads.square().sum([1,2,3])
                    loss_Dr1 = r1_penalty * (self.r1_gamma / 2)
                    training_stats.report('Loss/r1_penalty', r1_penalty)
                    training_stats.report('Loss/D/reg', loss_Dr1)

            with torch.autograd.profiler.record_function(name + '_backward'):
                (loss_Dreal + loss_Dr1).mean().mul(gain).backward()

#----------------------------------------------------------------------------
'''.strip()

loss_py.write_text(patched_loss_code + "\n", encoding="utf-8")
print("Patched loss.py with landmark regularization:", loss_py)

# =========================================================
# BUILD MODIFIED TRAIN CMD
# =========================================================
baseline_run = latest_run_dir(baseline_root)
baseline_snapshot = latest_snapshot(baseline_run) if baseline_run is not None else None

if baseline_snapshot is not None:
    resume_source = baseline_snapshot
    resume_mode = "resume_from_latest_baseline_snapshot"
else:
    resume_source = "https://api.ngc.nvidia.com/v2/models/nvidia/research/stylegan3/versions/1/files/stylegan3-r-ffhqu-256x256.pkl"
    resume_mode = "resume_from_pretrained_pickle"

def build_train_cmd(resume_source):
    return [
        sys.executable, str(train_py),
        "--outdir", str(modified_root),
        "--cfg", CFG,
        "--data", str(dataset_zip),
        "--gpus", "1",
        "--batch", str(BATCH),
        "--batch-gpu", str(BATCH_GPU),
        "--gamma", str(GAMMA),
        "--mirror", str(MIRROR),
        "--kimg", str(MOD_KIMG),
        "--tick", str(MOD_TICK),
        "--snap", str(MOD_SNAP),
        "--metrics", str(MOD_METRICS),
        "--seed", str(MOD_SEED),
        "--resume", str(resume_source),
        "--aug", str(AUG),
        "--cbase", str(CBASE),
        "--glr", str(GLR),
        "--dlr", str(DLR),
        "--workers", str(WORKERS),
        "--mbstd-group", str(MBSTD_GROUP),
    ]

env = os.environ.copy()
env["LM_REG_ENABLE"] = "1"
env["LM_REG_CKPT"] = str(reg_ckpt)
env["LM_REG_LAMBDA"] = str(LM_LAMBDA)
env["LM_REG_Z_MARGIN"] = str(LM_Z_MARGIN)

cmd = build_train_cmd(resume_source)

# =========================================================
# STATUS MODE
# =========================================================
if MODE.strip().lower() == "status":
    run_dir = latest_run_dir(modified_root)
    print("Latest modified run:", run_dir)
    if run_dir is None:
        print("No modified run found yet.")
    else:
        snap = latest_snapshot(run_dir)
        fake = latest_fakes(run_dir)
        stats_file = run_dir / "training_stats.jsonl"
        print("Latest snapshot:", snap)
        print("Latest fake grid:", fake)
        print("Stats file exists:", stats_file.exists())
        if stats_file.exists():
            print("\nLast few stat entries:")
            for row in tail_jsonl(stats_file, n=5):
                print(json.dumps(row, indent=2))

# =========================================================
# TRAIN MODE
# =========================================================
elif MODE.strip().lower() == "train":
    launch_info = {
        "time": datetime.now().isoformat(),
        "resume_mode": resume_mode,
        "resume_source": str(resume_source),
        "dataset_zip": str(dataset_zip),
        "stats_npz": str(stats_npz),
        "stats_csv": str(stats_csv),
        "landmark_regressor_ckpt": str(reg_ckpt),
        "lm_lambda": LM_LAMBDA,
        "lm_z_margin": LM_Z_MARGIN,
        "cfg": CFG,
        "batch": BATCH,
        "batch_gpu": BATCH_GPU,
        "gamma": GAMMA,
        "cbase": CBASE,
        "glr": GLR,
        "dlr": DLR,
        "aug": AUG,
        "mirror": MIRROR,
        "workers": WORKERS,
        "mbstd_group": MBSTD_GROUP,
        "mod_kimg": MOD_KIMG,
        "mod_tick": MOD_TICK,
        "mod_snap": MOD_SNAP,
        "mod_metrics": MOD_METRICS,
        "seed": MOD_SEED,
        "command": " ".join(str(x) for x in cmd),
    }

    launch_json = logs_dir / "landmark_reg_launcher_config.json"
    launch_json.write_text(json.dumps(launch_info, indent=2), encoding="utf-8")
    print("Saved launcher config:", launch_json)
    print(json.dumps(launch_info, indent=2))

    live_log = logs_dir / "landmark_reg_train_live.log"
    stream_process(cmd, cwd=stylegan3_repo, logfile=live_log, env=env)

    run_dir = latest_run_dir(modified_root)
    snap = latest_snapshot(run_dir) if run_dir else None
    fake = latest_fakes(run_dir) if run_dir else None

    print("\nModified training finished.")
    print("Run dir:", run_dir)
    print("Latest snapshot:", snap)
    print("Latest fake grid:", fake)
    print("Live log:", live_log)

else:
    raise ValueError("MODE must be 'train' or 'status'")

print("\nNotebook 07 complete.")

Feature names: ['eye_to_nose_ratio', 'mouth_width_ratio', 'nose_to_mouth_ratio', 'left_brow_to_eye_ratio', 'right_brow_to_eye_ratio', 'eye_symmetry', 'brow_symmetry', 'jaw_symmetry', 'face_hw_ratio']
Stats file: /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/data/manifests/real_landmark_stats.npz
CSV file: /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/data/manifests/real_landmark_features.csv
Regressor training device: cuda


Regressor train epoch 1/3:   0%|          | 0/71 [00:00<?, ?it/s]

Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x733b84728310><function _MultiProcessingDataLoaderIter.__del__ at 0x733b84728310>

Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/sajjan/.conda/envs/myenv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1479, in __del__
  File "/home/sajjan/.conda/envs/myenv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1479, in __del__
        self._shutdown_workers()self._shutdown_workers()

  File "/home/sajjan/.conda/envs/myenv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1462, in _shutdown_workers
  File "/home/sajjan/.conda/envs/myenv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1462, in _shutdown_workers
        if w.is_alive():if w.is_alive():

  File "/home/sajjan/.conda/envs/myenv/lib/python3.10/multiprocessing/process.py", line 160, in is_alive
  File "/home/sajjan/.conda/envs/

Epoch 1: train_loss=0.252577  val_loss=0.234423
Saved best regressor to: /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/checkpoints/aux_models/landmark_feature_regressor.pt


Regressor train epoch 2/3:   0%|          | 0/71 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x733b84728310><function _MultiProcessingDataLoaderIter.__del__ at 0x733b84728310>Exception ignored in: 
Traceback (most recent call last):

Traceback (most recent call last):
  File "/home/sajjan/.conda/envs/myenv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1479, in __del__
  File "/home/sajjan/.conda/envs/myenv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1479, in __del__
        self._shutdown_workers()self._shutdown_workers()

  File "/home/sajjan/.conda/envs/myenv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1462, in _shutdown_workers
      File "/home/sajjan/.conda/envs/myenv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1462, in _shutdown_workers
if w.is_alive():
  File "/home/sajjan/.conda/envs/myenv/lib/python3.10/multiprocessing/process.py", line 160, in is_alive
        assert self._parent_pid == os.getpid(), 'can o

Epoch 2: train_loss=0.248326  val_loss=0.234279
Saved best regressor to: /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/checkpoints/aux_models/landmark_feature_regressor.pt


Regressor train epoch 3/3:   0%|          | 0/71 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x733b84728310>
Exception ignored in: Traceback (most recent call last):
<function _MultiProcessingDataLoaderIter.__del__ at 0x733b84728310>  File "/home/sajjan/.conda/envs/myenv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1479, in __del__

    Traceback (most recent call last):
self._shutdown_workers()
  File "/home/sajjan/.conda/envs/myenv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1479, in __del__
  File "/home/sajjan/.conda/envs/myenv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1462, in _shutdown_workers
        self._shutdown_workers()if w.is_alive():

  File "/home/sajjan/.conda/envs/myenv/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1462, in _shutdown_workers
  File "/home/sajjan/.conda/envs/myenv/lib/python3.10/multiprocessing/process.py", line 160, in is_alive
        if w.is_alive():assert self._parent_pid == os.

Epoch 3: train_loss=0.245576  val_loss=0.230885
Saved best regressor to: /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/checkpoints/aux_models/landmark_feature_regressor.pt
Saved history: /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/logs/landmark_reg/landmark_regressor_history.json
Backed up original loss.py to: /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/third_party/stylegan3/training/loss_original_backup.py
Patched loss.py with landmark regularization: /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/third_party/stylegan3/training/loss.py
Saved launcher config: /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/logs/landmark_reg/landmark_reg_launcher_config.json
{
  "time": "2026-04-09T16:33:19.527265",
  "resume_mode": "resume_from_latest_baseline_snapshot",
  "resume_source": "/data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/checkpoints/baseline_stylegan3r/00005-stylegan3-r-celeba_256x256-gpus1-b